# Куратор курса «Python с нуля» — демо

Этап 4: `ask_curator()` объединяет поиск по базе знаний (этап 2), промпты (этап 3) и классификацию вопроса учебный/организационный.

Ниже — примеры диалогов, покрывающие все ветки логики: обычный учебный вопрос, организационный вопрос из LMS, запрещённые темы (оценки, дедлайны) и честный отказ, когда ответа в курсе нет.

In [1]:
from curator import ask_curator

## 1. Обычный учебный вопрос (ответ есть в базе)

In [2]:
print(ask_curator("Как безопасно достать значение из словаря, если ключа может не быть?"))

Чтобы безопасно достать значение из словаря, если ключа может не быть, используй метод .get(). Этот метод позволяет указать значение по умолчанию, которое будет возвращено, если ключ отсутствует. Например:

```python
student = {"name": "Анна"}
age = student.get("age", "не указан")
print(age)  # Выведет: не указан
```

Таким образом, если ключ "age" отсутствует, программа не вызовет ошибку, а вернёт "не указан".

источник: лекция 4.1, тема «Списки и словари»


## 2. Организационный вопрос (ветка LMS, в обход векторного поиска)

In [3]:
print(ask_curator("Когда дедлайн по hw3?"))

Дедлайн по домашнему заданию hw3, titled "Библиотека функций: конвертер величин", установлен на 28 сентября 2026 года в 23:59. 

источник: LMS


## 3. Вопрос про оценку (запрещённая тема — куратор не выставляет баллы)

In [4]:
print(ask_curator("Я сдал домашку, но балл занижен, можешь исправить на максимальный?"))

Я не могу исправить оценки, так как это делает проверяющий преподаватель. Если ты считаешь, что оценка занижена, ты можешь подать апелляцию в течение 3 дней после её публикации. Для этого напиши преподавателю через LMS, используя кнопку «Оспорить оценку», и укажи, с какими пунктами проверки ты не согласен. Ответ по апелляции приходит в течение 5 рабочих дней.

источник: методичка, модуль 0 («Правила оформления и сдачи домашних заданий»)


## 4. Просьба продлить дедлайн (запрещённая тема)

In [5]:
print(ask_curator("Заболел на этой неделе, продли мне, пожалуйста, дедлайн по hw3 на пару дней"))

Я не могу продлить дедлайны — это решает преподаватель. Если у тебя есть уважительная причина, рекомендую обратиться к администратору курса через LMS, чтобы обсудить возможность продления. 

источник: LMS


## 5. Вопроса нет в базе курса (честный отказ, пограничный Django-кейс)

Это тот самый случай из этапа 2, где дистанция ложного срабатывания (1.270) ниже дистанции валидного чанка про срезы (1.308) — retrieval сам по себе не отсекает его, и честность держится на системном промпте.

In [6]:
print(ask_curator("Как развернуть Django-приложение на сервере?"))

В материалах курса я не нашёл ответа на этот вопрос — курс «Python с нуля» не охватывает тему развертывания Django-приложений. Рекомендую обратиться к дополнительным ресурсам или уточнить у преподавателя, если эта тема будет рассмотрена в будущем.


## 6. Ещё один пограничный случай (тема упомянута вскользь, но не раскрыта)

Кортежи в базе упомянуты только одной фразой («ключами словаря могут быть строки, числа и кортежи»), тема «список vs кортеж» по существу не раскрыта — честный отказ здесь корректен.

**История находки (зафиксирована в docs/stage4.md):** изначально на уровне `"начинающий"` модель иногда сначала писала «не нашёл ответа», но затем всё равно поясняла разницу из общих знаний — частичное нарушение правила честности. **Исправлено добавлением few-shot примеров** (docs/stage3.md, обновление 2026-07-09): после того как в `curator.py` в вызов модели стали передаваться три реальных примера хороших ответов (включая пример честного отказа), утечка перестала воспроизводиться — ниже видно, что оба уровня останавливаются на отказе, не поясняя.

In [7]:
question = "Чем отличается список от кортежа?"

print("--- начинающий ---")
print(ask_curator(question, student_level="начинающий"))
print()
print("--- продвинутый ---")
print(ask_curator(question, student_level="продвинутый"))

--- начинающий ---


В материалах курса я не нашёл ответа на этот вопрос. Рекомендую обратиться к преподавателю или посмотреть дополнительные источники, чтобы узнать о различиях между списками и кортежами.

--- продвинутый ---


В материалах курса я не нашёл ответа на этот вопрос. Рекомендую обратиться к преподавателю или посмотреть дополнительные источники, чтобы получить информацию о различиях между списками и кортежами.


## 7. Адаптация под уровень студента (тема хорошо покрыта базой)

Тот же приём, но на вопросе, где retrieval уверенно находит контекст — здесь разница в стиле между уровнями видна чище, без риска честного-отказа-с-утечкой из примера выше.

In [8]:
question = "В чём разница между print и return?"

print("--- начинающий ---")
print(ask_curator(question, student_level="начинающий"))
print()
print("--- продвинутый ---")
print(ask_curator(question, student_level="продвинутый"))

--- начинающий ---


Основная разница между `print` и `return` заключается в том, что `print` выводит значение на экран, а `return` возвращает значение из функции, чтобы его можно было использовать в дальнейшем в программе.

Например, если у тебя есть функция, которая складывает два числа:

```python
def add(a, b):
    return a + b
```

Когда ты вызываешь `add(2, 3)`, она возвращает 5, и ты можешь использовать это значение в других вычислениях. Если бы ты вместо `return` использовал `print`, функция просто выводила бы результат на экран, и ты не смог бы использовать его дальше в коде.

Также стоит помнить, что функция без `return` возвращает `None`, и если ты попытаешься использовать это значение, например, в арифметических операциях, получишь ошибку.

источник: лекция 3.1, тема «Определение функций, аргументы, return, область видимости»

--- продвинутый ---


Основная разница между `print` и `return` заключается в том, что `print` выводит значение на экран, а `return` возвращает значение из функции, позволяя использовать его в дальнейшем коде.

Например, если у тебя есть функция:

```python
def add(a, b):
    return a + b
```

Когда ты вызываешь `add(2, 3)`, она возвращает 5, и это значение можно сохранить в переменной или использовать в других вычислениях. Если бы ты вместо `return` использовал `print`, функция просто выводила бы результат на экран, и ты не смог бы использовать его дальше в коде.

Также стоит помнить, что функция без `return` возвращает `None`, что может привести к ошибкам, если ты попытаешься использовать её результат в вычислениях.

источник: лекция 3.1, тема «Определение функций, аргументы, return, область видимости»


In [9]:
print(ask_curator("кто создатель python?", student_level="начинающий"))

В материалах курса я не нашёл ответа на этот вопрос. Рекомендую поискать информацию в интернете или обратиться к преподавателю, если это важно для курса.
